In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

from scripts.analytics import *
from scripts.algorithms import *


In [2]:
name = "adult"
namex = "Adult "

d = 14


###knn
graphs = np.load("../graphs/"+name+"_knn_random.npy", allow_pickle=True)
###threshold
graphsz = np.load("../graphs/"+name+"_thresh_random.npy", allow_pickle=True)


# Compute bruteforce algorithm results

## Nearest neighbor

In [3]:
### read the random, greedy results
df_greedy1 = pd.read_csv("./sm_results/"+name+"_knn_summary.csv")
df_greedy1.head()

,K,F(Sg+ U Sg-)*,(Sg+ U Sg-)*,F(Sr+ U Sr-)*,(Sr+ U Sr-)*,F(Sg),Sg,F(Sr),Sr,F(Sgfull),...,F(Sg)/F(Sr),F(Sg)/F(Sgfull),F(Sr)/F(Srfull),budgetGreedyTime,budgetRdmTime,kmax,n,m,dataset,graphid
0,1,70.0,set(),70.0,{14},70.0,set(),70.0,{44},70.0,...,1.0,1.0,1.0,0.007626,0.000169,1,328,46,adult (14),0
1,2,70.0,set(),70.0,"{1, 14}",70.0,set(),70.0,"{44, 7}",70.0,...,1.0,1.0,1.0,0.006177,0.000172,1,328,46,adult (14),0
2,3,70.0,set(),70.0,"{1, 14, 25}",70.0,set(),70.0,"{1, 44, 7}",70.0,...,1.0,1.0,1.0,0.006519,0.000172,1,328,46,adult (14),0
3,4,70.0,set(),70.0,"{1, 23, 14, 25}",70.0,set(),70.0,"{1, 44, 17, 7}",70.0,...,1.0,1.0,1.0,0.006394,0.000181,1,328,46,adult (14),0
4,5,70.0,set(),70.0,"{1, 22, 23, 25, 14}",70.0,set(),70.0,"{1, 7, 44, 15, 17}",70.0,...,1.0,1.0,1.0,0.005780,0.000174,1,328,46,adult (14),0


In [4]:
greedy_bruteforce_algosdf = pd.DataFrame()
all_bruteforce = pd.DataFrame()

for lx in range(len(graphs)):

    r_edges_random  = graphs[lx]["edges"]
    r_labels_random = graphs[lx]["labels"]
    
    optls = budget_opts(edges=r_edges_random, labels=r_labels_random, budget=5)
    df_optls = pd.DataFrame(optls)
    df_optls["graphidx"] = [lx]*len(df_optls)
    all_bruteforce = pd.concat([all_bruteforce, df_optls], ignore_index=True)


    df_greedy11 = df_greedy1[df_greedy1["graphid"]==lx][['K', 'Sg', 'F(Sg)', 'Sr', 'F(Sr)', '(Sg+ U Sg-)*',
                                                         'F(Sg+ U Sg-)*', '(Sr+ U Sr-)*', 'F(Sr+ U Sr-)*', 'kmax']]

    merged = pd.merge(df_greedy11, df_optls, left_on="K", right_on="b", how="inner").drop(columns=["b"])
    merged["F(Sg)/F(S*)"] = merged["F(Sg)"] / merged["F(S*)"]
    merged["F(Sr)/F(S*)"] = merged["F(Sr)"] / merged["F(S*)"]
    merged["F(Sg+ U Sg-)* / F(S*)"] = merged["F(Sg+ U Sg-)*"] / merged["F(S*)"]
    merged["F(Sr+ U Sr-)* / F(S*)"] = merged["F(Sr+ U Sr-)*"] / merged["F(S*)"]

    summary_dfv = merged[[
        "K", "Sg", "F(Sg)", "Sr", "F(Sr)", "(Sg+ U Sg-)*", "F(Sg+ U Sg-)*",
        "(Sr+ U Sr-)*", "F(Sr+ U Sr-)*", "F(S*)", "S*",
        "F(Sg)/F(S*)", "F(Sr)/F(S*)", "F(Sg+ U Sg-)* / F(S*)", "F(Sr+ U Sr-)* / F(S*)",
        "BFtime", "kmax"
    ]]

    summary_df = summary_dfv.copy()
    summary_df["n"] = [graphs[lx]['n']]*len(summary_df)
    summary_df["m"] = [graphs[lx]['m']]*len(summary_df)
    summary_df["dataset"] = [name+" ("+ str(d) + ")"]*len(summary_df)
    summary_df["graphid"] = [lx]*len(summary_df)

    
    greedy_bruteforce_algosdf = pd.concat([greedy_bruteforce_algosdf, summary_df], ignore_index=True)



## Thresholding

In [5]:
df_greedy1z = pd.read_csv("./sm_results/"+name+"_thresh_summary.csv")
df_greedy1z.head()

,K,F(Sg+ U Sg-)*,(Sg+ U Sg-)*,F(Sr+ U Sr-)*,(Sr+ U Sr-)*,F(Sg),Sg,F(Sr),Sr,F(Sgfull),...,F(Sg)/F(Sr),F(Sg)/F(Sgfull),F(Sr)/F(Srfull),budgetGreedyTime,budgetRdmTime,r,n,m,dataset,graphid
0,1,202.512964,{5},102.700900,{18},202.512964,{5},78.528840,{42},242.0,...,2.578836,0.836830,0.324499,0.039953,0.000764,4.0,328,48,adult (14),0
1,2,231.237085,"{9, 5}",203.679630,"{18, 5}",231.237085,"{9, 5}",78.528840,"{42, 7}",242.0,...,2.944614,0.955525,0.324499,0.076572,0.001012,4.0,328,48,adult (14),0
2,3,235.492857,"{8, 9, 5}",205.779630,"{18, 3, 5}",235.492857,"{8, 9, 5}",81.264813,"{1, 42, 7}",242.0,...,2.897845,0.973111,0.335805,0.127800,0.001258,4.0,328,48,adult (14),0
3,4,237.876190,"{8, 9, 5, 46}",234.503752,"{9, 18, 3, 5}",237.876190,"{8, 9, 5, 46}",81.868296,"{1, 42, 49, 7}",242.0,...,2.905596,0.982959,0.338299,0.170738,0.000794,4.0,328,48,adult (14),0
4,5,239.976190,"{3, 5, 8, 9, 46}",238.759524,"{18, 3, 5, 8, 9}",239.976190,"{3, 5, 8, 9, 46}",107.537818,"{1, 7, 42, 49, 18}",242.0,...,2.231552,0.991637,0.444371,0.198384,0.001273,4.0,328,48,adult (14),0


In [6]:
greedy_bruteforce_algosdfz = pd.DataFrame()
all_bruteforcez = pd.DataFrame()

for lz in range(len(graphsz)):

    r_edges_randomz  = graphsz[lz]["edges"]
    r_labels_randomz = graphsz[lz]["labels"]
    
    optlsz = budget_opts(edges=r_edges_randomz, labels=r_labels_randomz, budget=5)
    
    df_optlsz = pd.DataFrame(optlsz)
    df_optlsz["graphidx"] = [lz]*len(df_optlsz)
    all_bruteforcez = pd.concat([all_bruteforcez, df_optlsz], ignore_index=True)


    df_greedy11z = df_greedy1z[df_greedy1z["graphid"]==lz][['K', 'Sg', 'F(Sg)', 'Sr', 'F(Sr)', '(Sg+ U Sg-)*',
                                                         'F(Sg+ U Sg-)*', '(Sr+ U Sr-)*', 'F(Sr+ U Sr-)*', 'r']]

    mergedz = pd.merge(df_greedy11z, df_optlsz, left_on="K", right_on="b", how="inner").drop(columns=["b"])
    mergedz["F(Sg)/F(S*)"] = mergedz["F(Sg)"] / mergedz["F(S*)"]
    mergedz["F(Sr)/F(S*)"] = mergedz["F(Sr)"] / mergedz["F(S*)"]
    mergedz["F(Sg+ U Sg-)* / F(S*)"] = mergedz["F(Sg+ U Sg-)*"] / mergedz["F(S*)"]
    mergedz["F(Sr+ U Sr-)* / F(S*)"] = mergedz["F(Sr+ U Sr-)*"] / mergedz["F(S*)"]

    summary_dfzv = mergedz[[
        "K", "Sg", "F(Sg)", "Sr", "F(Sr)", "(Sg+ U Sg-)*", "F(Sg+ U Sg-)*",
        "(Sr+ U Sr-)*", "F(Sr+ U Sr-)*", "F(S*)", "S*",
        "F(Sg)/F(S*)", "F(Sr)/F(S*)", "F(Sg+ U Sg-)* / F(S*)", "F(Sr+ U Sr-)* / F(S*)",
        "BFtime", "r"
    ]]

    summary_dfz = summary_dfzv.copy()
    summary_dfz["n"] = [graphsz[lz]['n']]*len(summary_dfz)
    summary_dfz["m"] = [graphsz[lz]['m']]*len(summary_dfz)
    summary_dfz["dataset"] = [name+" ("+ str(d) + ")"]*len(summary_dfz)
    summary_dfz["graphid"] = [lz]*len(summary_dfz)

    
    greedy_bruteforce_algosdfz = pd.concat([greedy_bruteforce_algosdfz, summary_dfz], ignore_index=True)



# Save Results 

In [7]:
greedy_bruteforce_algosdf.head()

,K,Sg,F(Sg),Sr,F(Sr),(Sg+ U Sg-)*,F(Sg+ U Sg-)*,(Sr+ U Sr-)*,F(Sr+ U Sr-)*,F(S*),...,F(Sg)/F(S*),F(Sr)/F(S*),F(Sg+ U Sg-)* / F(S*),F(Sr+ U Sr-)* / F(S*),BFtime,kmax,n,m,dataset,graphid
0,1,set(),70.0,{44},70.0,set(),70.0,{14},70.0,70.0,...,1.0,1.0,1.0,1.0,0.006814,1,328,46,adult (14),0
1,2,set(),70.0,"{44, 7}",70.0,set(),70.0,"{1, 14}",70.0,70.0,...,1.0,1.0,1.0,1.0,0.141671,1,328,46,adult (14),0
2,3,set(),70.0,"{1, 44, 7}",70.0,set(),70.0,"{1, 14, 25}",70.0,70.0,...,1.0,1.0,1.0,1.0,2.057091,1,328,46,adult (14),0
3,4,set(),70.0,"{1, 44, 17, 7}",70.0,set(),70.0,"{1, 23, 14, 25}",70.0,70.0,...,1.0,1.0,1.0,1.0,25.051061,1,328,46,adult (14),0
4,5,set(),70.0,"{1, 7, 44, 15, 17}",70.0,set(),70.0,"{1, 22, 23, 25, 14}",70.0,70.0,...,1.0,1.0,1.0,1.0,202.071914,1,328,46,adult (14),0


In [8]:
all_bruteforce.head()

,b,F(S*),S*,BFtime,graphidx
0,0,70.0,{},0.000273,0
1,1,70.0,{},0.006814,0
2,2,70.0,{},0.141671,0
3,3,70.0,{},2.057091,0
4,4,70.0,{},25.051061,0


In [9]:
greedy_bruteforce_algosdfz.head()

,K,Sg,F(Sg),Sr,F(Sr),(Sg+ U Sg-)*,F(Sg+ U Sg-)*,(Sr+ U Sr-)*,F(Sr+ U Sr-)*,F(S*),...,F(Sg)/F(S*),F(Sr)/F(S*),F(Sg+ U Sg-)* / F(S*),F(Sr+ U Sr-)* / F(S*),BFtime,r,n,m,dataset,graphid
0,1,{5},202.512964,{42},78.528840,{5},202.512964,{18},102.700900,202.512964,...,1.0,0.387772,1.0,0.507132,0.037707,4.0,328,48,adult (14),0
1,2,"{9, 5}",231.237085,"{42, 7}",78.528840,"{9, 5}",231.237085,"{18, 5}",203.679630,231.237085,...,1.0,0.339603,1.0,0.880826,0.862474,4.0,328,48,adult (14),0
2,3,"{8, 9, 5}",235.492857,"{1, 42, 7}",81.264813,"{8, 9, 5}",235.492857,"{18, 3, 5}",205.779630,235.492857,...,1.0,0.345084,1.0,0.873825,14.260990,4.0,328,48,adult (14),0
3,4,"{8, 9, 5, 46}",237.876190,"{1, 42, 49, 7}",81.868296,"{8, 9, 5, 46}",237.876190,"{9, 18, 3, 5}",234.503752,237.876190,...,1.0,0.344163,1.0,0.985823,178.184871,4.0,328,48,adult (14),0
4,5,"{3, 5, 8, 9, 46}",239.976190,"{1, 7, 42, 49, 18}",107.537818,"{3, 5, 8, 9, 46}",239.976190,"{18, 3, 5, 8, 9}",238.759524,239.976190,...,1.0,0.448119,1.0,0.994930,1511.191182,4.0,328,48,adult (14),0


In [10]:
all_bruteforcez.head()

,b,F(S*),S*,BFtime,graphidx
0,0,76.838574,{},0.001589,0
1,1,202.512964,{5},0.037707,0
2,2,231.237085,"{9, 5}",0.862474,0
3,3,235.492857,"{8, 9, 5}",14.260990,0
4,4,237.876190,"{8, 9, 5, 46}",178.184871,0


In [11]:
greedy_bruteforce_algosdf.to_csv("./sm_results/"+name+"_knn_greedy_rdm_bruteforce.csv", index=False)
all_bruteforce.to_csv("./sm_results/"+name+"_knn_allbruteforce.csv", index=False)

greedy_bruteforce_algosdfz.to_csv("./sm_results/"+name+"_thresh_greedy_rdm_bruteforce.csv", index=False)
all_bruteforcez.to_csv("./sm_results/"+name+"_thresh_allbruteforce.csv", index=False)
